## Import Libraries

In [ ]:
# Load pandas for dataframe operations and numpy for numerical calculations

# Importing Libraries

import pandas as pd
import numpy as np

## Load Sales Data

In [ ]:
# Read tSales into a dataframe. One row per sale transaction across all brands and stores for the fiscal year

tSales = pd.read_csv("tSales - Copy.csv")

## Profile Sales Data

In [ ]:
# Confirm row count, column names, data types and check for null values before analysis begins

tSales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12697914 entries, 0 to 12697913
Data columns (total 7 columns):
 #   Column         Dtype  
---  ------         -----  
 0   salesID        int64  
 1   Store          int64  
 2   Brand          int64  
 3   SalesQuantity  int64  
 4   SalesPrice     float64
 5   SalesDate      object 
 6   ExciseTax      float64
dtypes: float64(2), int64(4), object(1)
memory usage: 678.1+ MB


## ADA Objective 1: Pricing Consistency

In [ ]:
# Group by Brand and calculate avg, min, max and count of SalesPrice
# Price range greater than zero flags brands sold at inconsistent prices across transactions

price_analysis = tSales.groupby('Brand').agg(
    avgPrice=('SalesPrice', 'mean'),
    minPrice=('SalesPrice', 'min'),
    maxPrice=('SalesPrice', 'max'),
    numTransactions=('SalesPrice', 'count')
)

price_analysis['priceRange'] = (
    price_analysis['maxPrice'] -
    price_analysis['minPrice']
)

display(price_analysis)

inconsistent_prices = price_analysis.loc[
price_analysis['priceRange'] > 0
]

display(inconsistent_prices)

,avgPrice,minPrice,maxPrice,numTransactions,priceRange
Brand,,,,,
58,12.990000,12.99,12.99,2095,0.0
60,10.622404,9.99,10.99,1148,1.0
61,13.990000,13.99,13.99,25,0.0
62,38.293595,36.99,41.99,2309,5.0
63,40.286296,38.99,43.99,1971,5.0
...,...,...,...,...,...
90089,119.990000,119.99,119.99,34,0.0
90090,649.990000,649.99,649.99,3,0.0
90604,119.990000,119.99,119.99,10,0.0


,avgPrice,minPrice,maxPrice,numTransactions,priceRange
Brand,,,,,
60,10.622404,9.99,10.99,1148,1.0
62,38.293595,36.99,41.99,2309,5.0
63,40.286296,38.99,43.99,1971,5.0
70,24.240000,18.99,24.99,8,6.0
72,36.680141,34.99,39.99,355,5.0
...,...,...,...,...,...
46985,23.745674,22.99,26.99,1498,4.0
47009,13.872039,12.99,15.99,1030,3.0
47011,19.590000,18.99,21.99,25,3.0


## Load Ending Inventory Data

In [ ]:
# Read tEndInv. Contains year end units on hand and purchase cost per brand per store. This is the cost side of the lower of cost or NRV comparison

tEndInv = pd.read_csv('tEndInv.csv')
display(tEndInv)

,ID,Store,Brand,onHand,PurchasePrice,endDate
0,1,1,58,12,12.99,2016-06-30
1,2,1,62,11,36.99,2016-06-30
2,3,1,63,3,38.99,2016-06-30
3,4,1,72,6,34.99,2016-06-30
4,5,1,75,9,14.99,2016-06-30
...,...,...,...,...,...,...
214206,214207,79,47090,8,23.99,2016-06-30
214207,214208,79,90011,12,144.99,2016-06-30
214208,214209,79,90089,24,119.99,2016-06-30
214209,214210,79,90609,23,24.99,2016-06-30


## Review Both Datasets

In [ ]:
# Display tSales and tEndInv side by side to confirm structure and verify the Brand and Store keys that will be used to merge them

display(tSales)
display(tEndInv)

,salesID,Store,Brand,SalesQuantity,SalesPrice,SalesDate,ExciseTax
0,1,1,10021,1,12.99,2015-07-12,0.11
1,2,1,10051,1,59.99,2015-07-28,0.11
2,3,1,10058,1,13.99,2015-07-02,0.11
3,4,1,10058,1,13.99,2015-07-04,0.11
4,5,1,10058,1,13.99,2015-07-06,0.11
...,...,...,...,...,...,...,...
12697909,12697910,9,966,2,20.99,2016-06-26,1.57
12697910,12697911,9,966,1,20.99,2016-06-29,0.79
12697911,12697912,9,984,1,25.99,2016-06-08,0.79
12697912,12697913,9,984,2,25.99,2016-06-17,1.57


,ID,Store,Brand,onHand,PurchasePrice,endDate
0,1,1,58,12,12.99,2016-06-30
1,2,1,62,11,36.99,2016-06-30
2,3,1,63,3,38.99,2016-06-30
3,4,1,72,6,34.99,2016-06-30
4,5,1,75,9,14.99,2016-06-30
...,...,...,...,...,...,...
214206,214207,79,47090,8,23.99,2016-06-30
214207,214208,79,90011,12,144.99,2016-06-30
214208,214209,79,90089,24,119.99,2016-06-30
214209,214210,79,90609,23,24.99,2016-06-30


## ADA Objective 2: Products Sold Below Cost

In [ ]:
# Left join tSales and tEndInv on Brand and Store to attach purchase cost to each transaction
# below_cost_sales shows transaction level flags
# below_cost aggregates to brand level and filters to brands where average selling price is below average cost indicating a sustained pricing problem

tEndInv = pd.read_csv('tEndInv.csv')

sales_inventory = tSales.merge(tEndInv, on=['Brand', 'Store'], how='left')

below_cost_sales = sales_inventory.loc[
    sales_inventory['SalesPrice'] < sales_inventory['PurchasePrice']
]

display(below_cost_sales)

below_cost = sales_inventory.groupby('Brand').agg(
    avgCost=('PurchasePrice', 'mean'),
    avgPrice=('SalesPrice', 'mean')
)

below_cost = below_cost.loc[
    below_cost['avgPrice'] < below_cost['avgCost']
]

display(below_cost)

,salesID,Store,Brand,SalesQuantity,SalesPrice,SalesDate,ExciseTax,ID,onHand,PurchasePrice,endDate
0,1,1,10021,1,12.99,2015-07-12,0.11,1815.0,24.0,13.99,2016-06-30
1,2,1,10051,1,59.99,2015-07-28,0.11,1816.0,8.0,63.99,2016-06-30
24,25,1,10236,2,14.99,2015-07-09,0.22,1820.0,50.0,15.99,2016-06-30
25,26,1,10236,1,14.99,2015-07-10,0.11,1820.0,50.0,15.99,2016-06-30
26,27,1,10236,1,14.99,2015-07-11,0.11,1820.0,50.0,15.99,2016-06-30
...,...,...,...,...,...,...,...,...,...,...,...
11615295,11615296,9,9388,1,23.99,2016-05-05,0.39,20881.0,14.0,25.99,2016-06-30
11615296,11615297,9,9388,1,23.99,2016-05-24,0.39,20881.0,14.0,25.99,2016-06-30
11638988,11638989,10,2986,1,65.99,2016-06-03,0.79,22421.0,5.0,69.99,2016-06-30
12119705,12119706,49,2986,1,65.99,2016-06-06,0.79,119039.0,10.0,69.99,2016-06-30


,avgCost,avgPrice
Brand,,
60,10.99,10.622404
70,24.99,24.240000
107,6.49,6.490000
126,32.99,31.550873
143,38.99,37.682308
...,...,...
46923,13.99,12.759231
46934,19.99,17.088206
46985,23.99,23.745674


## ADA Objective 3: Quantify Potential Misstatement

In [ ]:
# Aggregate both tables to brand level and merge
# Delta = average cost minus average selling price
# Potential misstatement = delta multiplied by units on hand
# Filter to brands where misstatement is positive meaning cost exceeds NRV

tEndInv_agg = tEndInv.groupby('Brand').agg(avgCost=('PurchasePrice', 'mean'), endingInventory=('onHand', 'sum'))
tSales_agg = tSales.groupby('Brand').agg(avgPrice=('SalesPrice', 'mean'))
inventory_analysis = pd.merge(tEndInv_agg, tSales_agg, on='Brand', how='left')

inventory_analysis['delta'] = (
    inventory_analysis['avgCost'] -
    inventory_analysis['avgPrice']
)

inventory_analysis['potentialMisstatement'] = (
    inventory_analysis['delta'] *
    inventory_analysis['endingInventory']
)

display(inventory_analysis)

misstatement_products = inventory_analysis.loc[
    inventory_analysis['potentialMisstatement'] > 0
]

display(misstatement_products)

,avgCost,endingInventory,avgPrice,delta,potentialMisstatement
Brand,,,,,
58,12.99,369,12.990000,0.000000,0.000000
60,10.99,31,10.622404,0.367596,11.395470
61,13.99,12,13.990000,0.000000,0.000000
62,36.99,466,38.293595,-1.303595,-607.475097
63,38.99,401,40.286296,-1.296296,-519.814815
...,...,...,...,...,...
90089,119.99,169,119.990000,0.000000,0.000000
90090,649.99,45,649.990000,0.000000,0.000000
90604,119.99,47,119.990000,0.000000,0.000000


,avgCost,endingInventory,avgPrice,delta,potentialMisstatement
Brand,,,,,
60,10.99,31,10.622404,3.675958e-01,1.139547e+01
70,24.99,5,24.240000,7.500000e-01,3.750000e+00
107,6.49,18,6.490000,8.881784e-16,1.598721e-14
126,32.99,3443,31.550873,1.439127e+00,4.954914e+03
143,38.99,9,37.682308,1.307692e+00,1.176923e+01
...,...,...,...,...,...
46762,13.99,1199,13.110295,8.797048e-01,1.054766e+03
46923,13.99,90,12.759231,1.230769e+00,1.107692e+02
46934,19.99,549,17.088206,2.901794e+00,1.593085e+03


## Profile Misstated Products

In [ ]:
# Confirm how many brands were flagged as potentially misstated before summing the total dollar amount

misstatement_products.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2503 entries, 60 to 90084
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   avgCost                2503 non-null   float64
 1   endingInventory        2503 non-null   int64  
 2   avgPrice               2503 non-null   float64
 3   delta                  2503 non-null   float64
 4   potentialMisstatement  2503 non-null   float64
dtypes: float64(4), int64(1)
memory usage: 117.3 KB


## Total Potential Misstatement

In [ ]:
# Sum potentialMisstatement across all flagged brands to produce the total inventory overstatement figure for the audit memo

misstatement_products['potentialMisstatement'].sum()

np.float64(1905046.6264929236)